# 02 · Model experiments

What was tried, and why the final setup looks the way it does. Everything here uses **cross-validation on the
development seasons (2012/13–2024/25)**. The ablation and the season-index experiment are re-run here (Ridge, fast);
grid search, interval coverage and test results are **read from `reports/`**, so no saved model is retrained
and the test season is not re-evaluated.

Prerequisites: `python -m src.data && python -m src.features && python -m src.train` (and `src.evaluate` for section 5).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data import TEST_SEASON
from src.features import FEATURES, FEATURES_PATH, RIDGE_EXTRA
from src.models import make_ridge
from src.train import cross_validate

pd.set_option("display.float_format", "{:,.3f}".format)
REPORTS = ROOT / "reports"
dev = pd.read_parquet(FEATURES_PATH).query("season < @TEST_SEASON")  # the test season is never used here
print(f"{len(dev):,} development rows, {len(FEATURES)} features")

24,930 development rows, 48 features


## 1. Model comparison (expanding-window CV, validation seasons 2021/22–2024/25)

`mean_bias` = mean(predicted − actual) log value. `overfit_gap` = CV RMSE − train RMSE.

In [2]:
cv = pd.read_csv(REPORTS / "cv_summary.csv", index_col="model")
cv[["label", "rmse_log", "r2_log", "median_ape", "within_50", "mean_bias", "train_rmse_log", "overfit_gap"]]

,label,rmse_log,r2_log,median_ape,within_50,mean_bias,train_rmse_log,overfit_gap
model,,,,,,,,
baseline_median,Median baseline (position x age band),1.167,0.079,0.670,0.336,-0.447,1.120,0.047
ridge_age_position,Ridge: age + position,1.143,0.117,0.666,0.354,-0.387,1.120,0.023
ridge,Ridge: all features,0.556,0.791,0.348,0.682,-0.051,0.578,-0.022
xgb,XGBoost: all features (main),0.502,0.829,0.307,0.739,-0.067,0.385,0.117
xgb_player_market,XGBoost: player-only + league market level,0.679,0.688,0.432,0.575,-0.048,0.560,0.119


## 2. What carries the signal? Ridge feature-group ablation

Drop one group at a time and measure how much CV RMSE rises. A single dominant group would hint at leakage;
instead the signal is spread over age, league, club strength, market level and playing time, while goals/assists
per 90 add little.

In [3]:
groups = {
    "market level (league S-1)": ["league_median_log_value_prev"],
    "club strength": ["club_ppg", "club_rank_pct"],
    "playing time": ["minutes", "appearances", "minutes_share"],
    "season per-90": ["goals_p90", "assists_p90", "ga_p90", "yellow_p90", "red_p90"],
    "previous season + form": [c for c in FEATURES if c.startswith(("prev_", "form_", "has_prev"))],
    "age / height": ["age", "age_sq", "height_in_cm"],
    "position one-hots": [c for c in FEATURES if c.startswith(("pos_", "subpos_"))],
    "league one-hots": [c for c in FEATURES if c.startswith("league_") and c != "league_median_log_value_prev"],
}

def cv_rmse(columns):
    return cross_validate(lambda: make_ridge(100.0, columns), dev)["rmse_log"].mean()

full = cv_rmse(FEATURES)
ablation = pd.DataFrame({"cv_rmse_without": {g: cv_rmse([c for c in FEATURES if c not in cols]) for g, cols in groups.items()}})
ablation["rmse_increase"] = ablation["cv_rmse_without"] - full
print(f"All features: CV RMSE {full:.4f}")
ablation.sort_values("rmse_increase", ascending=False)

All features: CV RMSE 0.5556


,cv_rmse_without,rmse_increase
age / height,0.819,0.264
club strength,0.707,0.152
market level (league S-1),0.659,0.104
playing time,0.634,0.079
previous season + form,0.601,0.045
league one-hots,0.568,0.012
season per-90,0.565,0.010
position one-hots,0.563,0.007


## 3. Market drift: season index vs lagged league market level

Market values roughly doubled between 2012/13 and 2024/25. The lagged league median (`league_median_log_value_prev`)
is in every model; adding a linear season trend on top does not lower RMSE and pushes the bias positive, so it was
not used.

In [4]:
rows = {}
for name, cols in [("Ridge, FEATURES", FEATURES), ("Ridge, FEATURES + season_index", FEATURES + RIDGE_EXTRA)]:
    folds = cross_validate(lambda c=cols: make_ridge(100.0, c), dev)
    rows[name] = folds.set_index("valid_season")["mean_bias"].to_dict() | {"mean RMSE": folds["rmse_log"].mean()}
pd.DataFrame(rows).T

,2021/22,2022/23,2023/24,2024/25,mean RMSE
"Ridge, FEATURES",-0.013,-0.081,-0.070,-0.041,0.556
"Ridge, FEATURES + season_index",0.108,0.069,0.078,0.114,0.556


## 4. XGBoost grid search and prediction intervals

The grid is flat near the top (the best configurations differ by about 0.002 RMSE), so the choice at the grid edge
is not hiding a much better model. Raw 10%/90% quantile intervals covered only ~74–75%; the conformal correction
(learned from earlier folds only) brings coverage to ~79–81%.

In [5]:
grid = pd.read_csv(REPORTS / "cv_grid_search.csv")
for model in ["xgb", "xgb_player_market"]:
    g = grid[grid["model"] == model]
    print(f"{model}: best {g['cv_rmse_log'].min():.4f}, 5th best {g['cv_rmse_log'].nsmallest(5).iloc[-1]:.4f}, "
          f"worst {g['cv_rmse_log'].max():.4f} over {len(g)} configs")
grid[grid["model"] == "xgb"].head(5)

xgb: best 0.5025, 5th best 0.5041, worst 0.5258 over 48 configs
xgb_player_market: best 0.6792, 5th best 0.6811, worst 0.7090 over 48 configs


,candidate,cv_rmse_log,model
10,"{'learning_rate': 0.05, 'max_depth': 5, 'min_c...",0.502,xgb
11,"{'learning_rate': 0.05, 'max_depth': 5, 'min_c...",0.503,xgb
12,"{'learning_rate': 0.1, 'max_depth': 3, 'min_ch...",0.504,xgb
13,"{'learning_rate': 0.1, 'max_depth': 3, 'min_ch...",0.504,xgb
14,"{'learning_rate': 0.05, 'max_depth': 5, 'min_c...",0.504,xgb


In [6]:
cv_folds = pd.read_csv(REPORTS / "cv_results.csv")
cv_folds[cv_folds["model"].isin(["xgb", "xgb_player_market"])][
    ["model", "valid_season", "coverage_raw", "conformal_correction", "coverage_conformal"]]

,model,valid_season,coverage_raw,conformal_correction,coverage_conformal
12,xgb,2021/22,0.736,NaN,NaN
13,xgb,2022/23,0.738,0.073,0.789
14,xgb,2023/24,0.752,0.082,0.814
15,xgb,2024/25,0.752,0.076,0.811
16,xgb_player_market,2021/22,0.721,NaN,NaN
17,xgb_player_market,2022/23,0.760,0.133,0.837
18,xgb_player_market,2023/24,0.749,0.097,0.813
19,xgb_player_market,2024/25,0.756,0.092,0.818


## 5. CV vs the one-time test evaluation (2025/26)

Read from `reports/test_metrics.csv`. RMSE matches CV for every model (no gap above 0.05), but the main model's bias
doubles to −0.15 (about −14%): the market rose faster than the lagged market level. That is why gaps and flags are
centred on each season's median.

In [7]:
test = pd.read_csv(REPORTS / "test_metrics.csv", index_col="model")
pd.DataFrame({"cv_rmse": cv["rmse_log"], "test_rmse": test["rmse_log"], "cv_bias": cv["mean_bias"],
              "test_bias": test["mean_bias"], "test_coverage": test["interval_coverage_80"]})

,cv_rmse,test_rmse,cv_bias,test_bias,test_coverage
model,,,,,
baseline_median,1.167,1.157,-0.447,-0.500,NaN
ridge_age_position,1.143,1.129,-0.387,-0.451,NaN
ridge,0.556,0.560,-0.051,-0.108,NaN
xgb,0.502,0.500,-0.067,-0.153,0.814
xgb_player_market,0.679,0.666,-0.048,-0.175,0.821


## Takeaways

- XGBoost beats Ridge (CV RMSE 0.502 vs 0.556) and both crush the baselines (1.14–1.17).
- The estimate is mostly **context-implied** (age, league, club strength, market level, playing time);
  goals/assists per 90 add little.
- Market drift is the main modelling problem: a lagged league market level fixes most of it in CV, but not all of
  it in the test season.
- Intervals needed a conformal correction to reach their nominal 80% coverage.